aug 17th update

In [14]:
import pandas as pd
import numpy as np
from scipy.interpolate import griddata

bathyyyy

In [15]:
db = pd.read_csv("..\\created_data\\gebco_bath.csv", header=None, skiprows=1,
                 names=['long', 'lat', 'bathy'])

In [16]:
db = db.iloc[::200]  # keeps 1 in every 200 rows, gebco super huge
print(db.shape)

(55296, 3)


In [13]:
dt = pd.read_csv("..\\data\\age_hf\\lat_long_age.csv", header=None, skiprows=1,
                 names=['lat', 'long', 'age'])
print(dt)

        lat    long       age
0     49.62 -129.98  0.816961
1     44.68 -125.29  9.119624
2     44.66 -125.27  9.119624
3     44.66 -125.27  9.119624
4     44.66 -125.26  9.119624
...     ...     ...       ...
7440  48.97 -122.10  9.284931
7441  48.97 -122.10  9.284931
7442  48.96 -122.09  9.284931
7443  51.92 -129.97  0.549429
7444  53.56 -131.43  7.047305

[7445 rows x 3 columns]


interpolate age at each bathy point (bc we have more bathy than age)

In [17]:
age_interp = griddata((dt[['long','lat']].values), dt['age'].values, db[['long','lat']].values, method='linear')
age_nan_fill = griddata((dt[['long','lat']].values), dt['age'].values, db[['long','lat']].values, method='nearest')

db['age'] = np.where(np.isnan(age_interp), age_nan_fill, age_interp)

print(db)

                long        lat   bathy        age
0        -132.997917  38.002083 -5025.0  33.939312
200      -132.164583  38.002083 -4836.0  33.939312
400      -131.331250  38.002083 -4911.0  33.939312
600      -130.497917  38.002083 -4759.0  30.501097
800      -129.664583  38.002083 -4679.0  35.199921
...              ...        ...     ...        ...
11058200 -125.164583  53.997917   744.0   5.251928
11058400 -124.331250  53.997917   819.0   5.871304
11058600 -123.497917  53.997917   655.0   6.490680
11058800 -122.664583  53.997917   571.0   7.110056
11059000 -121.831250  53.997917   678.0   7.339193

[55296 rows x 4 columns]


In [19]:
np.where(np.isnan(db))

(array([], dtype=int64), array([], dtype=int64))

In [ ]:
db[np.isclose(db['age'], 0.81696, atol=0.0005)]  

,long,lat,bathy,age
8038800,-129.997917,49.631250,-2385.0,0.816965
8053200,-129.997917,49.652083,-2336.0,0.816986
8067600,-129.997917,49.672917,-2280.0,0.817008
8260600,-129.831250,49.952083,-2134.0,0.817358
8410200,-130.497917,50.168750,-2402.0,0.816598
8730000,-129.997917,50.631250,-2120.0,0.816879


In [20]:
dg = pd.read_csv("..\\created_data\\FA.csv", header=None, skiprows=1, names=['long', 'lat', 'FA'])
print(dg)

             long        lat          FA
0     -133.000003  37.983334  -20.522749
1     -133.000003  38.100001  -24.824638
2     -133.000003  38.216667  -31.192137
3     -133.000003  38.333334  -38.186611
4     -133.000003  38.450001  -32.169643
...           ...        ...         ...
16693 -119.000003  53.500001  111.308601
16694 -119.000003  53.616668   50.057011
16695 -119.000003  53.733334   37.020630
16696 -119.000003  53.850001    4.265869
16697 -119.000003  53.966668  -10.105060

[16698 rows x 3 columns]


In [23]:
grav_interp = griddata((dg[['long','lat']].values), dg['FA'].values, db[['long','lat']].values, method='linear')
grav_nan_fill = griddata((dg[['long','lat']].values), dg['FA'].values, db[['long','lat']].values, method='nearest')

db['FA'] = np.where(np.isnan(grav_interp), grav_nan_fill, grav_interp)

print(db[3688:3689])

              long       lat   bathy        age         FA
737600 -131.664583  39.06875 -4537.0  33.939312 -16.864132


now bringing in heat flow

In [24]:
dq = pd.read_csv("..\\data\\age_hf\\ihfc_lat_long_q.csv", header=None, skiprows=1,
                 names=['lat', 'long', 'q_ihfc'])
print(dq)

        lat    long  q_ihfc
0     44.68 -125.29    79.0
1     44.66 -125.27    81.0
2     44.66 -125.27    93.0
3     44.66 -125.26    77.0
4     44.66 -125.25    81.0
...     ...     ...     ...
7462  48.97 -122.10   100.0
7463  48.97 -122.10    54.0
7464  48.96 -122.09   113.0
7465  51.92 -129.97    45.0
7466  53.56 -131.43    60.0

[7467 rows x 3 columns]


In [25]:
q_interp = griddata((dq[['long','lat']].values), dq['q_ihfc'].values, db[['long','lat']].values, method='linear')
q_nan_fill = griddata((dq[['long','lat']].values), dq['q_ihfc'].values, db[['long','lat']].values, method='nearest')

db['q_ihfc'] = np.where(np.isnan(q_interp), q_nan_fill, q_interp)

print(db[3688:3689])

              long       lat   bathy        age         FA  q_ihfc
737600 -131.664583  39.06875 -4537.0  33.939312 -16.864132    19.0


In [26]:
ps_q = []
stein_q = []
z_ps = []
z_stein = []

In [27]:
# pars-scl
for a in db['age']:
    z_ps.append(2500 + 350*(a**(0.5)))
    ps_q.append((11.3/(a**(0.5)))*41.868)  # μcal/cm^2*s --> mW/m^2

# stein x2
for a in db['age']:
    z_stein.append(2600+365*a**2)
    stein_q.append((510/a**(0.5)))

print(f'age: {db['age'][3688:3689].values}')
print(f'pars-scl depth: {z_ps[3688:3689]}, stein depth: {z_stein[3688:3689]}')
print(f'pars-scl heat flow: {ps_q[3688:3689]}, stein heat flow: {stein_q[3688:3689]}')

age: [33.939312]
pars-scl depth: [4539.010966130393], stein depth: [423035.06814717053]
pars-scl heat flow: [81.20993106488804], stein heat flow: [87.54244237281117]


In [28]:
db['q_ps'] = ps_q
db['q_stein'] = stein_q

db['ps_z'] = z_ps
db['stein_z'] = z_stein

In [30]:
db[np.isclose(db['age'], 0.81696, atol=0.0005)]  

,long,lat,bathy,age,FA,q_ihfc,q_ps,q_stein,ps_z,stein_z
8038800,-129.997917,49.631250,-2385.0,0.816965,-1.018680,334.720974,523.430272,564.245824,2816.351477,2843.612829
8053200,-129.997917,49.652083,-2336.0,0.816986,1.294621,310.423221,523.423513,564.238537,2816.355563,2843.625413
8067600,-129.997917,49.672917,-2280.0,0.817008,5.355937,286.125468,523.416754,564.231251,2816.359648,2843.637998
8260600,-129.831250,49.952083,-2134.0,0.817358,0.418818,129.449438,523.304589,564.110341,2816.427456,2843.846949
8410200,-130.497917,50.168750,-2402.0,0.816598,-0.666514,657.582741,523.548023,564.372756,2816.280327,2843.393740
8730000,-129.997917,50.631250,-2120.0,0.816879,-27.807050,114.678545,523.458028,564.275743,2816.334704,2843.561165


In [31]:
db.shape

(55296, 10)

In [ ]:
train_filt_dg = db[db['long'].between(-130, -128) & db['lat'].between(44, 46)]
test_filt_dg = db[db['long'].between(-130, -128) & db['lat'].between(46, 48)]

# test_filt_dg = dg[dg['long'].between(-132, -128) & dg['lat'].between(46, 50)]

In [32]:
db.to_csv("..\\created_data\\lola_bathy_age_q_grav_z.csv", index=False)

In [34]:
train_filt_dg.to_csv("..\\created_data\\train_filt_lola_grav_age_q_z_bath.csv", index=False)
test_filt_dg.to_csv("..\\created_data\\test_filt_lola_grav_age_q_z_bath.csv", index=False)